In [1]:
import os
import shutil
from google.colab import drive

# Remove existing directory if it exists
mount_point = '/content/drive'
if os.path.exists(mount_point):
    shutil.rmtree(mount_point)

# Create fresh directory
os.makedirs(mount_point)

# Now mount
drive.mount('/content/drive')

!pip install pyheif

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 5.7 MB/s eta 0:00:00


In [ ]:
"""
Recapture Detector (ResNet50) — Patch-based training (Fully fixed & complete)
Experiment 16 | Patch-based for local pattern detection | All outputs saved
Now processes BOTH datasets:
1. NTU-Roselab-Dataset (flat originals / recaptures)
2. Devices folder with 5 mobile devices: ["Iphone-11","Iphone-16", "Nothing_2a", "Pixel","Poco-M3"]
All images merged into single originals / recaptures pool for training
"""

# Step 1: Imports
import os
import json
import random
import hashlib
import time
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import applications, layers, models, callbacks
from PIL import Image
import pyheif
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Step 2: Settings & Hyperparameters
EXPERIMENT = 16
DATA_ROOT_NTU = "/content/drive/MyDrive/NTU-Roselab-Dataset"
DATA_ROOT_DEVICES = "/content/drive/MyDrive/Devices"
MERGED = "/content/merged_combined_datasets"
OUTPUT_DIR = "/content/drive/MyDrive/Recapture_Photo_Detection/Resnet50_Combined/results"
SPLIT_DIR = "/content/drive/MyDrive/Recapture_Photo_Detection"
IMG_SIZE = 224
LARGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 1e-4
N_FOLDS = 3
MODEL_NAME = "Resnet50"
PREPROCESSING = "none"
DROPOUT_RATE = 0.4
SEED = 42
TRAIN_FRAC, TEST_FRAC = 0.85, 0.15
REPEAT_FACTOR = 4
NUM_TEST_PATCHES = 5

DEVICES = ["Iphone-11","Iphone-16", "Nothing_2a", "Pixel","Poco-M3"]

HYPERPARAMETERS = {
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "epochs": EPOCHS,
    "n_folds": N_FOLDS,
    "dropout_rate": DROPOUT_RATE,
    "large_size": LARGE_SIZE,
    "repeat_factor": REPEAT_FACTOR,
    "num_test_patches": NUM_TEST_PATCHES,
    "datasets": ["NTU-Roselab-Dataset", "Devices (5 mobiles)"]
}

# Step 3: Create directories
for p in [OUTPUT_DIR, f"{OUTPUT_DIR}/checkpoints", MERGED, SPLIT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

# Step 4: Image conversion utilities
def ensure_jpg_from_path(src_path: Path, dst_path: Path, quality=95):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if dst_path.exists():
        return
    sfx = src_path.suffix.lower()
    try:
        if sfx in {".heic", ".heif"}:
            heif_file = pyheif.read(str(src_path))
            img = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data,
                                  "raw", heif_file.mode, heif_file.stride)
            img = img.convert("RGB")
            img.save(dst_path, "JPEG", quality=quality)
        else:
            img = Image.open(src_path).convert("RGB")
            img.save(dst_path, "JPEG", quality=quality)
    except Exception as e:
        print(f"[WARN] Failed converting {src_path}: {e}")

# Generic copy function (with optional prefix for device name)
def copy_or_convert(src_folder, dst_folder, prefix=""):
    cnt = 0
    src = Path(src_folder)
    if not src.exists():
        print(f"[INFO] Folder not found: {src_folder}")
        return 0
    dst = Path(dst_folder)
    for f in src.rglob("*.*"):
        if f.suffix.lower() in {".jpg",".jpeg",".png",".heic",".heif"}:
            name = f"{prefix}_{f.name}".rsplit(".",1)[0] + ".jpg" if prefix else f"{f.stem}.jpg"
            dst_path = dst / name
            ensure_jpg_from_path(f, dst_path)
            cnt += 1
    return cnt

# Step 5: Merge & convert from BOTH datasets
print("\n--- MERGING & CONVERTING FROM BOTH DATASETS ---")
os.makedirs(f"{MERGED}/originals", exist_ok=True)
os.makedirs(f"{MERGED}/recaptures", exist_ok=True)

total_orig_cnt = 0
total_recap_cnt = 0

# 1. NTU-Roselab Dataset (flat structure)
ntu_orig_cnt = copy_or_convert(f"{DATA_ROOT_NTU}/originals", f"{MERGED}/originals", "ntu")
ntu_recap_cnt = copy_or_convert(f"{DATA_ROOT_NTU}/recaptures", f"{MERGED}/recaptures", "ntu")
total_orig_cnt += ntu_orig_cnt
total_recap_cnt += ntu_recap_cnt
print(f"NTU-Roselab → Originals: {ntu_orig_cnt} | Recaptures: {ntu_recap_cnt}")

# 2. Devices dataset (5 mobiles)
for dev in DEVICES:
    orig_cnt = copy_or_convert(f"{DATA_ROOT_DEVICES}/{dev}/originals", f"{MERGED}/originals", dev.lower())
    recap_cnt = copy_or_convert(f"{DATA_ROOT_DEVICES}/{dev}/recaptures", f"{MERGED}/recaptures", dev.lower())
    total_orig_cnt += orig_cnt
    total_recap_cnt += recap_cnt
    print(f"{dev} → Originals: {orig_cnt} | Recaptures: {recap_cnt}")

print(f"TOTAL Converted originals: {total_orig_cnt} | recaptures: {total_recap_cnt}")

if total_orig_cnt + total_recap_cnt == 0:
    print("[ERROR] No images found or converted from either dataset. Stopping pipeline.")
    sys.exit(1)

# Step 6: Collect paths
def get_paths(folder):
    return sorted([str(p) for p in Path(folder).glob("*.jpg")])

orig_paths = get_paths(f"{MERGED}/originals")
recap_paths = get_paths(f"{MERGED}/recaptures")
print(f"Final - Originals: {len(orig_paths)} | Recaptures: {len(recap_paths)}")

print(f"[INFO] No. images processed: Originals {len(orig_paths)}, Recaptures {len(recap_paths)}, Total {len(orig_paths)+len(recap_paths)}")

# Step 7: Shuffle and split (class-balanced)
np.random.seed(SEED)
random.seed(SEED)
np.random.shuffle(orig_paths)
np.random.shuffle(recap_paths)

def split_list(lst, train_frac):
    n = len(lst)
    if n == 0:
        return [], []
    n_train = int(np.floor(train_frac * n))
    return lst[:n_train], lst[n_train:]

train_o, test_o = split_list(orig_paths, TRAIN_FRAC)
train_r, test_r = split_list(recap_paths, TRAIN_FRAC)

train_paths = train_o + train_r
test_paths = test_o + test_r
train_labels = [0] * len(train_o) + [1] * len(train_r)
test_labels = [0] * len(test_o) + [1] * len(test_r)

print(f"TRAIN → Original: {len(train_o)} | Recapture: {len(train_r)} | Total: {len(train_paths)}")
print(f"TEST  → Original: {len(test_o)} | Recapture: {len(test_r)} | Total: {len(test_paths)}")

if len(train_paths) == 0:
    print("[ERROR] No training images after split. Stopping.")
    sys.exit(1)

print(f"[INFO] Train images: {len(train_paths)} | Test images: {len(test_paths)}")

# Step 8: Assurance checks
assert len(set(train_paths) & set(test_paths)) == 0, "Overlap detected!"
print("No train/test overlap — GOOD.")

# Save split hashes
def path_hash(p): return hashlib.md5(p.encode()).hexdigest()
split_hashes = {
    'train_hashes': [path_hash(p) for p in sorted(train_paths)],
    'test_hashes': [path_hash(p) for p in sorted(test_paths)]
}
with open(f"{SPLIT_DIR}/split_hashes_combined.json","w") as f:
    json.dump(split_hashes, f, indent=2)

# Step 9: Dataset characteristics (with source info via prefix)
def source_count(paths):
    counts = {"ntu": 0, "devices": 0}
    device_detail = {}
    for p in paths:
        stem = Path(p).stem
        if stem.startswith("ntu"):
            counts["ntu"] += 1
        else:
            device = stem.split('_')[0]
            device_detail[device] = device_detail.get(device, 0) + 1
            counts["devices"] += 1
    return counts, device_detail

orig_source, orig_detail = source_count(orig_paths)
recap_source, recap_detail = source_count(recap_paths)

dataset_info = {
    'total_originals': len(orig_paths),
    'total_recaptures': len(recap_paths),
    'total_images': len(orig_paths)+len(recap_paths),
    'sources_originals': orig_source,
    'sources_recaptures': recap_source,
    'device_detail_originals': orig_detail,
    'device_detail_recaptures': recap_detail,
    'splits': {
        'train': {'originals': len(train_o), 'recaptures': len(train_r)},
        'test': {'originals': len(test_o), 'recaptures': len(test_r)}
    }
}
with open(f"{OUTPUT_DIR}/dataset_characteristics.json","w") as f:
    json.dump(dataset_info, f, indent=2)
pd.DataFrame([dataset_info]).to_csv(f"{OUTPUT_DIR}/dataset_characteristics.csv", index=False)
print("Saved dataset characteristics (including source breakdown).")

# Step 10: Pipeline summary
pipeline_summary = {
    'experiment': f"Experiment - {EXPERIMENT}",
    'datasets_used': ["NTU-Roselab-Dataset", "Devices (5 mobiles)"],
    'seed': SEED,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'repeat_factor': REPEAT_FACTOR,
    'augmentation': ['random_flip_left_right','random_brightness(0.1)','random_contrast(0.9,1.1)'],
    'preprocessing': PREPROCESSING,
    'model': 'ResNet50(include_top=False) + GAP + Dropout(0.4) + Dense(128,relu) + Dense(1,sigmoid)',
    'optimizer': f'Adam({LEARNING_RATE})',
    'loss': 'binary_crossentropy',
    'metrics': ['accuracy'],
    'split': {'train_frac': TRAIN_FRAC, 'test_frac': TEST_FRAC},
    'hyperparameters': HYPERPARAMETERS
}
with open(f"{OUTPUT_DIR}/pipeline_summary.json","w") as f:
    json.dump(pipeline_summary, f, indent=2)
with open(f"{OUTPUT_DIR}/README_pipeline.md","w") as f:
    f.write("# Pipeline Summary - Combined Datasets\n\n")
    for k,v in pipeline_summary.items():
        f.write(f"- **{k}**: {v}\n")
print("Saved pipeline summary.")

# Step 11: TF Dataset helpers
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def decode_and_resize(path, resize_size=LARGE_SIZE):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (resize_size, resize_size))
    return img

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img

@tf.function
def process_train(path_tensor, label):
    img = decode_and_resize(path_tensor)
    if tf.random.uniform(()) < 0.7:
        img = augment(img)
    img = tf.image.random_crop(img, (IMG_SIZE, IMG_SIZE, 3))
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return img, label

@tf.function
def process_val_test(path_tensor, label):
    img = decode_and_resize(path_tensor)
    offset = (LARGE_SIZE - IMG_SIZE) // 2
    img = tf.image.crop_to_bounding_box(img, offset, offset, IMG_SIZE, IMG_SIZE)
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return img, label

# Step 12: Test dataset
test_paths_tf = tf.constant(test_paths, dtype=tf.string)
test_labels_tf = tf.constant(test_labels, dtype=tf.int32)
test_ds = tf.data.Dataset.from_tensor_slices((test_paths_tf, test_labels_tf))
test_ds = test_ds.map(process_val_test, num_parallel_calls=tf.data.AUTOTUNE) \
                 .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Step 13: Class weights
unique_labels = np.unique(train_labels)
if len(unique_labels) == 1:
    print(f"[INFO] Only one class found. Using uniform weight.")
    class_weight_dict = {int(unique_labels[0]): 1.0}
else:
    weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels)
    class_weight_dict = {int(k): float(v) for k, v in zip(unique_labels, weights)}
print(f"Class weights: {class_weight_dict}")

# Step 14: Cross-validation training
print("\n--- STARTING 5-FOLD CROSS-VALIDATION ---")
start_time = time.time()

EXPERIMENT_NAME = f"exp_{EXPERIMENT}_{MODEL_NAME}_combined_patch"
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []
best_val_acc = 0.0
best_fold = 0
best_model = None

train_paths_np = np.array(train_paths)
train_labels_np = np.array(train_labels)

for fold_no, (subtrain_idx, val_idx) in enumerate(skf.split(train_paths_np, train_labels_np), 1):
    print(f"\n=== Fold {fold_no}/{N_FOLDS} ===")
    subtrain_paths = train_paths_np[subtrain_idx].tolist()
    subtrain_labels = train_labels_np[subtrain_idx].tolist()
    val_paths = train_paths_np[val_idx].tolist()
    val_labels = train_labels_np[val_idx].tolist()

    print(f"[INFO] Fold {fold_no} - Subtrain images: {len(subtrain_paths)} | Val images: {len(val_paths)}")

    # Training dataset
    train_ds = tf.data.Dataset.from_tensor_slices((tf.constant(subtrain_paths), tf.constant(subtrain_labels)))
    train_ds = train_ds.repeat(REPEAT_FACTOR) \
                       .shuffle(buffer_size=8000, seed=SEED, reshuffle_each_iteration=True) \
                       .map(process_train, num_parallel_calls=tf.data.AUTOTUNE) \
                       .batch(BATCH_SIZE) \
                       .prefetch(tf.data.AUTOTUNE)

    # Validation dataset
    val_ds = tf.data.Dataset.from_tensor_slices((tf.constant(val_paths), tf.constant(val_labels)))
    val_ds = val_ds.map(process_val_test, num_parallel_calls=tf.data.AUTOTUNE) \
                   .batch(BATCH_SIZE) \
                   .prefetch(tf.data.AUTOTUNE)

    # Model
    base = applications.ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)

    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    ckpt_path = f"{OUTPUT_DIR}/checkpoints/{EXPERIMENT_NAME}_fold{fold_no}.weights.h5"
    cp = callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, save_weights_only=True,
                                   monitor='val_accuracy', mode='max', verbose=1)
    es = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1)
    lr_reduce = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

    if Path(ckpt_path).exists():
        print("Resuming from existing checkpoint...")
        model.load_weights(ckpt_path)

    # Train
    steps_per_epoch = max(1, (len(subtrain_paths) * REPEAT_FACTOR) // BATCH_SIZE)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                        steps_per_epoch=steps_per_epoch,
                        callbacks=[cp, es, lr_reduce],
                        class_weight=class_weight_dict,
                        verbose=1)

    # Save history & curves
    hist_df = pd.DataFrame(history.history)
    hist_df.to_csv(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_history.csv", index=False)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss Curve'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(1,2,2)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title('Accuracy Curve'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)
    plt.savefig(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_learning_curve.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Multi-patch prediction
    def predict_multi_patch(paths, labels, model, num_patches=NUM_TEST_PATCHES):
        crops = [
            (0, 0),
            (0, LARGE_SIZE - IMG_SIZE),
            (LARGE_SIZE - IMG_SIZE, 0),
            (LARGE_SIZE - IMG_SIZE, LARGE_SIZE - IMG_SIZE),
            ((LARGE_SIZE - IMG_SIZE)//2, (LARGE_SIZE - IMG_SIZE)//2)
        ][:num_patches]
        y_true = []
        y_pred_probs = []
        for path, label in zip(paths, labels):
            img = decode_and_resize(tf.constant(path))
            img = tf.expand_dims(img, 0)
            probs = []
            for cy, cx in crops:
                patch = img[:, cy:cy+IMG_SIZE, cx:cx+IMG_SIZE, :]
                prob = float(model(patch, training=False)[0][0])
                probs.append(prob)
            avg_prob = np.mean(probs)
            y_true.append(label)
            y_pred_probs.append(avg_prob)
        y_pred = [1 if p > 0.5 else 0 for p in y_pred_probs]
        return np.array(y_true), np.array(y_pred)

    y_val_true, y_val_pred = predict_multi_patch(val_paths, val_labels, model)
    y_test_true, y_test_pred = predict_multi_patch(test_paths, test_labels, model)

    cm_val = confusion_matrix(y_val_true, y_val_pred)
    cm_test = confusion_matrix(y_test_true, y_test_pred)

    acc_val = round(np.sum(y_val_true == y_val_pred) / len(y_val_true), 2) if len(y_val_true) > 0 else 0.0
    acc_test = round(np.sum(y_test_true == y_test_pred) / len(y_test_true), 2) if len(y_test_true) > 0 else 0.0

    # Save confusion matrices
    for split, cm, acc in [("val", cm_val, acc_val), ("test", cm_test, acc_test)]:
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Original','Recaptured'], yticklabels=['Original','Recaptured'])
        plt.title(f"Fold {fold_no} {split.upper()} CM | Acc: {acc:.2f}")
        plt.savefig(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_cm_{split}.png", dpi=300, bbox_inches='tight')
        plt.close()
        pd.DataFrame(cm).to_csv(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_cm_{split}.csv", index=False)

    # Save reports
    report_val = classification_report(y_val_true, y_val_pred, target_names=['Original','Recaptured'], output_dict=True, zero_division=0)
    report_test = classification_report(y_test_true, y_test_pred, target_names=['Original','Recaptured'], output_dict=True, zero_division=0)
    pd.DataFrame(report_val).transpose().round(2).to_csv(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_report_val.csv")
    pd.DataFrame(report_test).transpose().round(2).to_csv(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_report_test.csv")

    # Fold summary
    fold_summary = {
        'fold': fold_no,
        'val_accuracy': acc_val,
        'test_accuracy': acc_test
    }
    with open(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_fold{fold_no}_summary.json","w") as f:
        json.dump(fold_summary, f, indent=2)

    fold_results.append(fold_summary)

    if acc_val > best_val_acc:
        best_val_acc = acc_val
        best_fold = fold_no
        best_model = model

# Step 15: Final results
avg_val_acc = round(np.mean([f['val_accuracy'] for f in fold_results]), 2)
avg_test_acc = round(np.mean([f['test_accuracy'] for f in fold_results]), 2)
print(f"\nAverage Validation Accuracy: {avg_val_acc:.2f} | Average Test Accuracy: {avg_test_acc:.2f}")

avg_summary = {
    'experiment': EXPERIMENT_NAME,
    'avg_val_accuracy': avg_val_acc,
    'avg_test_accuracy': avg_test_acc,
    'fold_details': fold_results
}
with open(f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_average_summary.json","w") as f:
    json.dump(avg_summary, f, indent=2)

# Step 16: Save best model & TFLite
if best_model:
    best_model_path = f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_best_fold{best_fold}.keras"
    best_model.save(best_model_path)
    print(f"Saved best model: {best_model_path}")

    converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite = converter.convert()
    tflite_path = f"{OUTPUT_DIR}/{EXPERIMENT_NAME}_detector.tflite"
    Path(tflite_path).write_bytes(tflite)
    print(f"Saved TFLite model: {tflite_path}")

# Step 17: Duration
duration = time.time() - start_time
print(f"\nTotal duration till completion: {duration:.2f} seconds ({duration/60:.2f} minutes)")

print("\nALL DONE. Results saved to:", OUTPUT_DIR)
print("This model was trained on BOTH NTU-Roselab and 5 mobile devices datasets combined.")


--- MERGING & CONVERTING FROM BOTH DATASETS ---
NTU-Roselab → Originals: 1200 | Recaptures: 1199
Iphone-11 → Originals: 30 | Recaptures: 30
Iphone-16 → Originals: 40 | Recaptures: 27
Nothing_2a → Originals: 105 | Recaptures: 103
Pixel → Originals: 100 | Recaptures: 100
Poco-M3 → Originals: 133 | Recaptures: 131
TOTAL Converted originals: 1608 | recaptures: 1590
Final - Originals: 1182 | Recaptures: 1367
[INFO] No. images processed: Originals 1182, Recaptures 1367, Total 2549
TRAIN → Original: 1004 | Recapture: 1161 | Total: 2165
TEST  → Original: 178 | Recapture: 206 | Total: 384
[INFO] Train images: 2165 | Test images: 384
No train/test overlap — GOOD.
Saved dataset characteristics (including source breakdown).
Saved pipeline summary.
Class weights: {0: 1.078187250996016, 1: 0.9323858742463393}

--- STARTING 5-FOLD CROSS-VALIDATION ---

=== Fold 1/3 ===
[INFO] Fold 1 - Subtrain images: 1443 | Val images: 722
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/8
360/360 ━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_accuracy improved from 0.78947 to 0.79224, saving model to /content/drive/MyDrive/Recapture_Photo_Detection/Resnet50_Combined/results/checkpoints/exp_14_Resnet50_combined_patch_fold1.weights.h5
360/360 ━━━━━━━━━━━━━━━━━━━━ 117s 249ms/step - accuracy: 0.9167 - loss: 0.3214 - val_accuracy: 0.7922 - val_loss: 0.6172 - learning_rate: 1.0000e-04
Epoch 3/8
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 968ms/step - accuracy: 0.9417 - loss: 0.1555
Epoch 3: val_accuracy improved from 0.79224 to 0.81025, saving model to /content/drive/MyDrive/Recapture_Photo_Detection/Resnet50_Combined/results/checkpoints/exp_14_Resnet50_combined_patch_fold1.weights.h5
360/360 ━━━━━━━━━━━━━━━━━━━━ 401s 1s/step - accuracy: 0.9417 - loss: 0.1555 - val_accuracy: 0.8102 - val_loss: 0.7384 - learning_rate: 1.0000e-04
Epoch 4/8
  1/360 ━━━━━━━━━━━━━━━━━━━━ 1:06 184ms/step - accuracy: 0.9167 - loss: 0.1334
Epoch 4: val_accuracy improved from 0.81025 to 0.81579, saving model to /content/drive/MyDrive/Recapture_Photo_Det

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_accuracy improved from 0.77147 to 0.77424, saving model to /content/drive/MyDrive/Recapture_Photo_Detection/Resnet50_Combined/results/checkpoints/exp_14_Resnet50_combined_patch_fold2.weights.h5
360/360 ━━━━━━━━━━━━━━━━━━━━ 83s 159ms/step - accuracy: 0.9167 - loss: 0.1251 - val_accuracy: 0.7742 - val_loss: 0.6827 - learning_rate: 1.0000e-04
Epoch 3/8
360/360 ━━━━━━━━━━━━━━━━━━━━ 0s 958ms/step - accuracy: 0.9416 - loss: 0.1574
Epoch 3: val_accuracy improved from 0.77424 to 0.79086, saving model to /content/drive/MyDrive/Recapture_Photo_Detection/Resnet50_Combined/results/checkpoints/exp_14_Resnet50_combined_patch_fold2.weights.h5
360/360 ━━━━━━━━━━━━━━━━━━━━ 439s 1s/step - accuracy: 0.9416 - loss: 0.1573 - val_accuracy: 0.7909 - val_loss: 0.8636 - learning_rate: 1.0000e-04
Epoch 4/8
  1/360 ━━━━━━━━━━━━━━━━━━━━ 1:04 181ms/step - accuracy: 0.9167 - loss: 0.2514
Epoch 4: val_accuracy improved from 0.79086 to 0.79640, saving model to /content/drive/MyDrive/Recapture_Photo_Dete